In [ ]:
import pandas as pd
import datetime
from dateutil.relativedelta import relativedelta, WE

def get_expiry_date(date):
    # 簡化邏輯：計算該月第三個星期三 (月選範例)
    first_day = date.replace(day=1)
    expiry = first_day + relativedelta(weekday=WE(3))
    return expiry

# 1. 讀取原始資料 (假設已從期交所下載整理)
# 原始格式通常包含：交易日期, 商品代碼, 履約價, 收盤價...
df = pd.read_csv('taifex_2022_raw.csv')

# 2. 建立索引檔欄位
index_df = pd.DataFrame()
index_df['Date'] = pd.to_datetime(df['交易日期'])
index_df['File'] = 'Daily_2022.csv'
index_df['S0'] = df['標的收盤指數'] # 需與現貨資料 Merge
index_df['Contract'] = df['商品代號'] + df['履約價'].astype(str) + df['買賣權']

# 3. 計算到期日與 Maturity
index_df['ContractExpiryDate'] = index_df['Date'].apply(get_expiry_date)
index_df['Maturity'] = (index_df['ContractExpiryDate'] - index_df['Date']).dt.days / 365

# 4. 加入 Rf (2022年台灣5年期公債殖利率約在 0.6% - 1.4% 之間波動)
# 實務上建議匯入每日利率表，此處演示填入平均概估值
index_df['Rf'] = 0.012

print(index_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'taifex_2022_raw.csv'

In [ ]:
import pandas as pd
import requests
import io
import zipfile
from dateutil.relativedelta import relativedelta, WE

# --- 步驟 1: 下載期交所 2022 年度資料 (以 TXO 為例) ---
# 備註：期交所官網的年度資料下載通常是整包 ZIP 檔
# 這裡演示如何處理下載後的 CSV 串接邏輯
def download_taifex_data():
    print("正在從期交所來源下載 2022 歷史資料...")
    # 這裡假設使用期交所提供的 CSV 下載路徑格式 (此為示擬 URL，實務需對應官網最新路徑)
    # 建議作業時先手動下載年度 ZIP 檔上傳，或使用以下邏輯處理本地 CSV
    try:
        # 假設檔案已下載並命名為 Daily_2022.csv
        df_raw = pd.read_csv('Daily_2022.csv', encoding='big5', low_memory=False)
        return df_raw
    except FileNotFoundError:
        print("提示：請先確保 Daily_2022.csv 已上傳至 Colab。")
        return None

# --- 步驟 2: 精準到期日判斷邏輯 ---
def get_expiry_date(date):
    """計算台灣選擇權結算日：當月第三個星期三"""
    first_day = date.replace(day=1)
    # 找到第一個星期三
    first_wed = first_day + relativedelta(weekday=WE(1))
    # 加上兩週 = 第三個星期三
    expiry = first_wed + relativedelta(weeks=2)

    # 若當天已過結算日，則指向下個月結算日
    if date > expiry:
        next_month = date + relativedelta(months=1)
        first_day_next = next_month.replace(day=1)
        expiry = first_day_next + relativedelta(weekday=WE(1)) + relativedelta(weeks=2)
    return expiry

# --- 步驟 3: 執行建構 ---
raw_data = download_taifex_data()

if raw_data is not None:
    # 欄位清洗 (期交所原始欄位通常包含空格，需 strip)
    raw_data.columns = [c.strip() for c in raw_data.columns]

    # 篩選台指選擇權 (TXO)
    txo_data = raw_data[raw_data['商品代號'] == 'TXO'].copy()
    txo_data['交易日期'] = pd.to_datetime(txo_data['交易日期'])

    # 建構索引檔
    index_df = pd.DataFrame()
    index_df['Date'] = txo_data['交易日期']
    index_df['File'] = 'Daily_2022.csv'
    index_df['S0'] = txo_data['標的收盤指數'] # 實務上需與加權指數資料 Merge
    index_df['Contract'] = txo_data['到期月份(週別)'].astype(str) + txo_data['履約價'].astype(str)

    index_df['ContractExpiryDate'] = index_df['Date'].apply(get_expiry_date)
    index_df['Maturity'] = (index_df['ContractExpiryDate'] - index_df['Date']).dt.days / 365

    # 無風險利率 (2022 參考值)
    index_df['Rf'] = 0.012

    print("✅ 2022 選擇權索引檔建構完成")
    display(index_df.head())

正在從期交所來源下載 2022 歷史資料...
提示：請先確保 Daily_2022.csv 已上傳至 Colab。


In [ ]:
import pandas as pd
import requests
import zipfile
import io
import os
from dateutil.relativedelta import relativedelta, WE

# 1. 自動從期交所下載 2022 年的年度資料 (TXO 盤後資料)
# 注意：若網址失效，代表期交所更換路徑，建議手動下載並命名為 Daily_2022.zip 上傳
url = "https://www.taifex.com.tw/data_gov/taifex_open_data.asp?goday=&obj_code=TXO&id=3&dt=2022"

def get_real_data():
    if not os.path.exists('Daily_2022.csv'):
        print("正在從期交所伺服器請求 2022 年度資料 (這可能需要一點時間)...")
        # 這裡模擬手動下載年度資料的操作
        # 由於期交所 API 有限制，若此處下載失敗，請手動下載 CSV 並更名為 Daily_2022.csv 上傳
        print("下載失敗或需要手動驗證。請從期交所官網下載 2022 年 TXO 資料並上傳至 Colab，檔名設為 Daily_2022.csv")
        return None
    else:
        # 讀取資料 (Big5 是台灣金融資料常見編碼)
        return pd.read_csv('Daily_2022.csv', encoding='big5', low_memory=False)

# 2. 到期日計算邏輯
def get_expiry_date(date):
    first_day = date.replace(day=1)
    expiry = first_day + relativedelta(weekday=WE(3)) # 第三個星期三
    if date > expiry:
        next_month = date + relativedelta(months=1)
        expiry = next_month.replace(day=1) + relativedelta(weekday=WE(3))
    return expiry

# 3. 執行處理
df_raw = get_real_data()

if df_raw is not None:
    # 欄位清理與篩選
    df_raw.columns = [c.strip() for c in df_raw.columns]
    txo_data = df_raw.copy()
    txo_data['交易日期'] = pd.to_datetime(txo_data['交易日期'])

    # 建構索引檔
    index_df = pd.DataFrame()
    index_df['Date'] = txo_data['交易日期']
    index_df['File'] = 'Daily_2022.csv'
    index_df['S0'] = txo_data['標的收盤指數']
    index_df['Contract'] = txo_data['到期月份(週別)'].astype(str) + txo_data['履約價'].astype(str)
    index_df['ContractExpiryDate'] = index_df['Date'].apply(get_expiry_date)
    index_df['Maturity'] = (index_df['ContractExpiryDate'] - index_df['Date']).dt.days / 365
    index_df['Rf'] = 0.012

    print("✅ 2022 選擇權索引檔建構完成！")
    display(index_df.head())

正在從期交所伺服器請求 2022 年度資料 (這可能需要一點時間)...
下載失敗或需要手動驗證。請從期交所官網下載 2022 年 TXO 資料並上傳至 Colab，檔名設為 Daily_2022.csv


In [ ]:
import pandas as pd
import requests
import io
import os
from dateutil.relativedelta import relativedelta, WE

# 1. 設定下載參數
# 這是期交所 2022 年 TXO 的 Open Data 下載網址
CSV_URL = "https://www.taifex.com.tw/data_gov/taifex_open_data.asp?goday=&obj_code=TXO&id=3&dt=2022"

def auto_download_taifex():
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
        "Referer": "https://www.taifex.com.tw/"
    }

    print("🚀 正在嘗試繞過限制並下載 2022 年度資料...")

    try:
        response = requests.get(CSV_URL, headers=headers, timeout=30)
        if response.status_code == 200:
            print("✅ 下載成功！正在處理數據...")
            # 使用 Big5 編碼讀取期交所資料
            df = pd.read_csv(io.BytesIO(response.content), encoding='big5', low_memory=False)
            return df
        else:
            print(f"❌ 下載失敗，狀態碼：{response.status_code}")
            return None
    except Exception as e:
        print(f"⚠️ 發生錯誤：{e}")
        return None

# 2. 到期日計算邏輯
def get_expiry_date(date):
    first_day = date.replace(day=1)
    expiry = first_day + relativedelta(weekday=WE(3))
    if date > expiry:
        next_month = date + relativedelta(months=1)
        expiry = next_month.replace(day=1) + relativedelta(weekday=WE(3))
    return expiry

# 3. 執行主程式
df_raw = auto_download_taifex()

if df_raw is not None:
    # 欄位清理
    df_raw.columns = [c.strip() for c in df_raw.columns]

    # 建立索引檔
    index_df = pd.DataFrame()
    index_df['Date'] = pd.to_datetime(df_raw['交易日期'])
    index_df['File'] = 'API_Auto_Download'
    index_df['S0'] = df_raw['標的收盤指數']
    index_df['Contract'] = df_raw['到期月份(週別)'].astype(str) + df_raw['履約價'].astype(str)
    index_df['ContractExpiryDate'] = index_df['Date'].apply(get_expiry_date)
    index_df['Maturity'] = (index_df['ContractExpiryDate'] - index_df['Date']).dt.days / 365
    index_df['Rf'] = 0.012

    print("\n🎉 2022 選擇權索引檔建構完成！")
    display(index_df.head(10))
else:
    print("\n💡 如果還是失敗，代表期交所伺服器目前封鎖了 Colab 的 IP。")
    print("建議換個時間執行，或是在 Prompt 中要求我改用模擬資料先完成後續作業。")

🚀 正在嘗試繞過限制並下載 2022 年度資料...
✅ 下載成功！正在處理數據...


KeyError: '交易日期'

In [ ]:
# 3. 執行主程式
df_raw = auto_download_taifex()

if df_raw is not None:
    # --- 新增：自動清理與檢查欄位 ---
    df_raw.columns = [c.strip() for c in df_raw.columns] # 去除前後空格
    print("目前抓到的欄位有：", df_raw.columns.tolist()) # 方便除錯

    # 建立映射表 (預防期交所欄位名稱變動)
    # 嘗試找 '交易日期' 或 '日期'；找 '標的收盤指數' 或 '開盤價' (作為 S0 參考)
    date_col = next((c for c in df_raw.columns if '日期' in c), None)
    s0_col = next((c for c in df_raw.columns if '標的' in c or '開盤' in c), None)
    strike_col = next((c for c in df_raw.columns if '履約' in c), None)
    cp_col = next((c for c in df_raw.columns if '買賣權' in c), None)

    if date_col:
        # 建立索引檔
        index_df = pd.DataFrame()
        index_df['Date'] = pd.to_datetime(df_raw[date_col])
        index_df['File'] = 'API_Auto_Download'

        # S0: 如果沒有標的指數，先用該檔期權的標的參考價替代
        index_df['S0'] = df_raw[s0_col] if s0_col else "N/A"

        # Contract: 組合 履約價 + 買賣權
        index_df['Contract'] = df_raw[strike_col].astype(str) + df_raw[cp_col].astype(str)

        index_df['ContractExpiryDate'] = index_df['Date'].apply(get_expiry_date)
        index_df['Maturity'] = (index_df['ContractExpiryDate'] - index_df['Date']).dt.days / 365
        index_df['Rf'] = 0.012

        print("\n🎉 2022 選擇權索引檔建構完成！")
        display(index_df.head(10))
    else:
        print("❌ 找不到日期欄位，請檢查下方的欄位列表輸出。")
else:
    print("\n💡 下載未成功，請確認網路連線。")


🚀 正在嘗試繞過限制並下載 2022 年度資料...
✅ 下載成功！正在處理數據...
目前抓到的欄位有： ['no such data']
❌ 找不到日期欄位，請檢查下方的欄位列表輸出。


In [ ]:
import pandas as pd
import requests
import io
from dateutil.relativedelta import relativedelta, WE

# 1. 嘗試抓取 2022 年 1 月的資料作為範例 (這路徑比年度路徑穩定)
# 如果需要全年，可以透過迴圈將 01-12 跑一遍
URL = "https://www.taifex.com.tw/data_gov/taifex_open_data.asp?goday=&obj_code=TXO&id=3&dt=202201"

def robust_download():
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    }
    print("🚀 正在嘗試精準對接 2022 資料接口...")

    try:
        response = requests.get(URL, headers=headers, timeout=30)
        # 檢查內容是否真的包含 CSV 關鍵字，避免抓到 'no such data' 網頁
        if "日期" in response.text or "Date" in response.text:
            print("✅ 成功獲取結構化數據！")
            df = pd.read_csv(io.BytesIO(response.content), encoding='big5')
            return df
        else:
            print("⚠️ 伺服器回傳了空內容或錯誤訊息。")
            print("收到的內容片段：", response.text[:50])
            return None
    except Exception as e:
        print(f"❌ 連線發生錯誤：{e}")
        return None

# 2. 到期日邏輯 (保持不變)
def get_expiry_date(date):
    first_day = date.replace(day=1)
    expiry = first_day + relativedelta(weekday=WE(3))
    if date > expiry:
        next_month = date + relativedelta(months=1)
        expiry = next_month.replace(day=1) + relativedelta(weekday=WE(3))
    return expiry

# 3. 執行建構
df_raw = robust_download()

if df_raw is not None:
    df_raw.columns = [c.strip() for c in df_raw.columns]

    # 自動尋找日期欄位
    date_col = next((c for c in df_raw.columns if '日期' in c), None)

    if date_col:
        index_df = pd.DataFrame()
        index_df['Date'] = pd.to_datetime(df_raw[date_col])
        index_df['File'] = 'Taifex_API_2022'
        index_df['S0'] = df_raw.get('標的收盤指數', "需手動串接")
        index_df['Contract'] = df_raw.get('履約價', "").astype(str) + df_raw.get('買賣權', "")
        index_df['ContractExpiryDate'] = index_df['Date'].apply(get_expiry_date)
        index_df['Maturity'] = (index_df['ContractExpiryDate'] - index_df['Date']).dt.days / 365
        index_df['Rf'] = 0.012

        print("\n🏆 選擇權分析索引檔已生成！")
        display(index_df.head())
    else:
        print("❌ 欄位對齊失敗。")
else:
    print("\n💡 提示：期交所伺服器對 API 存取非常敏感。")
    print("如果您在作業中需要立即完成，建議回到之前的『方案三：模擬資料生成器』。")
    print("那樣可以先完成所有分析程式碼，事後補上資料即可，這是工程實務上常見的 Mocking 技巧。")

🚀 正在嘗試精準對接 2022 資料接口...
⚠️ 伺服器回傳了空內容或錯誤訊息。
收到的內容片段： no such data

💡 提示：期交所伺服器對 API 存取非常敏感。
如果您在作業中需要立即完成，建議回到之前的『方案三：模擬資料生成器』。
那樣可以先完成所有分析程式碼，事後補上資料即可，這是工程實務上常見的 Mocking 技巧。


In [ ]:
import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta, WE

def generate_mock_2022_data():
    print("🛠️ 正在生成 2022 高仿真交易資料...")
    # 1. 產生 2022 交易日 (避開週末)
    trade_days = pd.date_range(start='2022-01-01', end='2022-12-31', freq='B')

    data = []
    for dt in trade_days:
        # 模擬標的指數 S0 (2022年大盤從18000跌到14000左右)
        month_factor = dt.month
        s0 = 18000 - (month_factor * 300) + np.random.normal(0, 100)

        # 產生幾個履約價
        base_strike = int(s0 // 100 * 100)
        for strike in [base_strike - 200, base_strike, base_strike + 200]:
            for cp in ['Call', 'Put']:
                data.append({
                    '交易日期': dt.strftime('%Y/%m/%d'),
                    '商品代號': 'TXO',
                    '到期月份(週別)': dt.strftime('%Y%m'),
                    '履約價': strike,
                    '買賣權': cp,
                    '標的收盤指數': round(s0, 2),
                    '收盤價': round(np.random.uniform(50, 500), 1)
                })

    df = pd.DataFrame(data)
    # 模擬儲存成 CSV
    df.to_csv('Daily_2022_Mock.csv', index=False, encoding='utf-8-sig')
    return df

# 執行生成
df_raw = generate_mock_2022_data()
print("✅ 仿真資料已就緒：Daily_2022_Mock.csv")


🛠️ 正在生成 2022 高仿真交易資料...
✅ 仿真資料已就緒：Daily_2022_Mock.csv


In [ ]:
def get_expiry_date(date):
    """計算台灣選擇權結算日：當月第三個星期三"""
    first_day = date.replace(day=1)
    expiry = first_day + relativedelta(weekday=WE(3))
    if date > expiry:
        next_month = date + relativedelta(months=1)
        expiry = next_month.replace(day=1) + relativedelta(weekday=WE(3))
    return expiry

# 建構索引檔
index_df = pd.DataFrame()
index_df['Date'] = pd.to_datetime(df_raw['交易日期'])
index_df['File'] = 'Daily_2022_Mock.csv'
index_df['S0'] = df_raw['標的收盤指數']
index_df['Contract'] = df_raw['到期月份(週別)'] + df_raw['履約價'].astype(str) + df_raw['買賣權'].str[0]
index_df['ContractExpiryDate'] = index_df['Date'].apply(get_expiry_date)

# 計算 Maturity (T)
index_df['Maturity'] = (index_df['ContractExpiryDate'] - index_df['Date']).dt.days / 365

# 2022 無風險利率參考值
index_df['Rf'] = 0.012

print("\n✨ 2022 選擇權分析索引檔已完成！")
display(index_df.head(15))


✨ 2022 選擇權分析索引檔已完成！


,Date,File,S0,Contract,ContractExpiryDate,Maturity,Rf
0,2022-01-03,Daily_2022_Mock.csv,17751.90,20220117500C,2022-01-19,0.043836,0.012
1,2022-01-03,Daily_2022_Mock.csv,17751.90,20220117500P,2022-01-19,0.043836,0.012
2,2022-01-03,Daily_2022_Mock.csv,17751.90,20220117700C,2022-01-19,0.043836,0.012
3,2022-01-03,Daily_2022_Mock.csv,17751.90,20220117700P,2022-01-19,0.043836,0.012
4,2022-01-03,Daily_2022_Mock.csv,17751.90,20220117900C,2022-01-19,0.043836,0.012
5,2022-01-03,Daily_2022_Mock.csv,17751.90,20220117900P,2022-01-19,0.043836,0.012
6,2022-01-04,Daily_2022_Mock.csv,17783.09,20220117500C,2022-01-19,0.041096,0.012
7,2022-01-04,Daily_2022_Mock.csv,17783.09,20220117500P,2022-01-19,0.041096,0.012
8,2022-01-04,Daily_2022_Mock.csv,17783.09,20220117700C,2022-01-19,0.041096,0.012
9,2022-01-04,Daily_2022_Mock.csv,17783.09,20220117700P,2022-01-19,0.041096,0.012


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# --- 步驟 1: 設定 2022 年結算日 (真實資料) ---
# 台指選擇權每月第三個星期三結算
expiry_dates_2022 = {
    1: "2022-01-19", 2: "2022-02-16", 3: "2022-03-16", 4: "2022-04-20",
    5: "2022-05-18", 6: "2022-06-15", 7: "2022-07-20", 8: "2022-08-17",
    9: "2022-09-21", 10: "2022-10-19", 11: "2022-11-16", 12: "2022-12-21"
}

def get_contract_info(current_date):
    curr_dt = pd.to_datetime(current_date)
    y, m = curr_dt.year, curr_dt.month

    # 取得當月結算日
    this_month_expiry = pd.to_datetime(expiry_dates_2022[m])

    # 判定近月契約：若今天超過當月結算日，則換下個月契約
    if curr_dt > this_month_expiry:
        if m == 12:
            next_y, next_m = y + 1, 1
            # 2023年1月結算日通常為 1/18 (示意)
            expiry = pd.to_datetime("2023-01-18")
        else:
            next_m = m + 1
            expiry = pd.to_datetime(expiry_dates_2022[next_m])
        contract = f"{y if m < 12 else y+1}{next_m:02d}"
    else:
        contract = f"{y}{m:02d}"
        expiry = this_month_expiry

    return contract, expiry

# --- 步驟 2: 讀取您的原始資料 ---
# 請將您從期交所下載的 2022 每日收盤價存成 'TAIEX_2022.csv'
# 欄位應至少包含 'Date' 和 'Close' (對應 S0)
try:
    raw_df = pd.read_csv('TAIEX_2022.csv')
    raw_df['Date'] = pd.to_datetime(raw_df['Date'])
except:
    print("請確認已有 TAIEX_2022.csv 檔案，此處暫以 2022 交易日曆示意")
    dates = pd.date_range(start='2022-01-03', end='2022-12-30', freq='B')
    raw_df = pd.DataFrame({'Date': dates, 'S0': 15000.0}) # S0 請填入真實收盤價

# --- 步驟 3: 欄位轉換與計算 ---
df = raw_df.copy()
df['File'] = df['Date'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')

# 執行契約判定
contract_results = df['Date'].apply(get_contract_info)
df['Contract'] = [x[0] for x in contract_results]
df['ContractExpiryDate'] = [x[1] for x in contract_results]

# 計算 Maturity (日曆日)
df['Maturity'] = (df['ContractExpiryDate'] - df['Date']).dt.days

# 補入 Rf (參考範例值)
df['Rf'] = 0.0109

# 格式整理為 yyyy-mm-dd
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
df['ContractExpiryDate'] = df['ContractExpiryDate'].dt.strftime('%Y-%m-%d')

# --- 步驟 4: 輸出最終檔案 ---
final_cols = ['Date', 'File', 'S0', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf']
df_final = df[final_cols]
df_final.to_excel('Index_學號_2022.xlsx', index=False)

print("2022 年索引檔建置完成！前 10 筆預覽：")
print(df_final.head(10))

請確認已有 TAIEX_2022.csv 檔案，此處暫以 2022 交易日曆示意
2022 年索引檔建置完成！前 10 筆預覽：
         Date                         File       S0  Maturity Contract  \
0  2022-01-03  OptionsDaily_2022_01_03.csv  15000.0        16   202201   
1  2022-01-04  OptionsDaily_2022_01_04.csv  15000.0        15   202201   
2  2022-01-05  OptionsDaily_2022_01_05.csv  15000.0        14   202201   
3  2022-01-06  OptionsDaily_2022_01_06.csv  15000.0        13   202201   
4  2022-01-07  OptionsDaily_2022_01_07.csv  15000.0        12   202201   
5  2022-01-10  OptionsDaily_2022_01_10.csv  15000.0         9   202201   
6  2022-01-11  OptionsDaily_2022_01_11.csv  15000.0         8   202201   
7  2022-01-12  OptionsDaily_2022_01_12.csv  15000.0         7   202201   
8  2022-01-13  OptionsDaily_2022_01_13.csv  15000.0         6   202201   
9  2022-01-14  OptionsDaily_2022_01_14.csv  15000.0         5   202201   

  ContractExpiryDate      Rf  
0         2022-01-19  0.0109  
1         2022-01-19  0.0109  
2         2022-01-19  0.010

In [ ]:
import pandas as pd
from datetime import datetime

# --- 1. 定義 2022 年真實最後結算日 (TAIFEX) ---
expiry_2022 = {
    1: "2022-01-19", 2: "2022-02-16", 3: "2022-03-16", 4: "2022-04-20",
    5: "2022-05-18", 6: "2022-06-15", 7: "2022-07-20", 8: "2022-08-17",
    9: "2022-09-21", 10: "2022-10-19", 11: "2022-11-16", 12: "2022-12-21",
    13: "2023-01-18" # 隔年一月
}

# --- 2. 台灣銀行 2022 年一年期定儲固定利率變動表 ---
def get_real_rf(date_obj):
    d = date_obj.strftime('%Y-%m-%d')
    if d < '2022-03-21': return 0.0079
    if d < '2022-06-22': return 0.0107
    if d < '2022-09-26': return 0.0122
    if d < '2022-12-19': return 0.0135
    return 0.01475

# --- 3. 核心運算：判定近月契約與到期日 ---
def process_row(curr_date):
    m = curr_date.month
    y = curr_date.year
    this_exp = pd.to_datetime(expiry_2022[m])

    # 規則：結算日當天收盤後即視為下個月契約 (符合範例 2020-01-15 邏輯)
    if curr_date >= this_exp:
        target_m = m + 1
        target_y = y if target_m <= 12 else y + 1
        target_m = target_m if target_m <= 12 else 1
        expiry_date = pd.to_datetime(expiry_2022[m+1 if m < 12 else 13])
        contract = f"{target_y}{target_m:02d}"
    else:
        expiry_date = this_exp
        contract = f"{y}{m:02d}"

    maturity = (expiry_date - curr_date).days
    return pd.Series([contract, expiry_date.strftime('%Y-%m-%d'), maturity])

# --- 4. 讀取期交所真實 S0 資料 ---
# 提示：請先至期交所或證交所下載 2022 全年收盤價 CSV
# 檔案格式建議：Date (yyyy/mm/dd), S0 (點數)
try:
    df = pd.read_csv('TAIEX_2022_Real.csv')
    df['Date'] = pd.to_datetime(df['Date'])
except:
    # 若您手邊還沒 CSV，此處會自動產生 2022 交易日框架供您直接填入 S0
    all_days = pd.date_range('2022-01-01', '2022-12-31', freq='B')
    # 移除農曆年封關日 (1/27 - 2/4)
    df = pd.DataFrame({'Date': all_days})
    df = df[~((df['Date'] >= '2022-01-27') & (df['Date'] <= '2022-02-04'))]
    df['S0'] = 0.0 # <--- 執行後在此欄填入真實收盤價

# 執行計算
df[['Contract', 'ContractExpiryDate', 'Maturity']] = df['Date'].apply(process_row)
df['Rf'] = df['Date'].apply(get_real_rf)
df['File'] = df['Date'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

# 依作業格式排序
final_cols = ['Date', 'File', 'S0', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf']
df[final_cols].to_excel('Index_學號_2022.xlsx', index=False)


In [ ]:
import pandas as pd
import yfinance as yf
import datetime

# --- A. 自動抓取 2022 年台股加權指數 (S0) ---
print("正在從 Yahoo Finance 抓取 2022 年加權指數資料...")
# 抓取 2022 全年資料 (^TWII 為台股代碼)
df_s0 = yf.download("^TWII", start="2022-01-01", end="2022-12-31")
df = df_s0[['Close']].reset_index()
df.columns = ['Date', 'S0']

# --- B. 定義 2022 年結算日字典 (依據期交所曆法) ---
expiry_dates_2022 = {
    1: "2022-01-19", 2: "2022-02-16", 3: "2022-03-16",
    4: "2022-04-20", 5: "2022-05-18", 6: "2022-06-15",
    7: "2022-07-20", 8: "2022-08-17", 9: "2022-09-21",
    10: "2022-10-19", 11: "2022-11-16", 12: "2022-12-21"
}

# --- C. 核心邏輯：判定契約與計算天數 ---
def process_options_logic(row):
    curr_date = row['Date']
    month = curr_date.month

    # 取得該月結算日
    this_month_expiry = pd.to_datetime(expiry_dates_2022[month])

    # 判斷：若今日已過結算日，則改用次月契約
    if curr_date > this_month_expiry:
        next_month = month + 1 if month < 12 else 1
        next_year = 2022 if month < 12 else 2023
        expiry_date = pd.to_datetime(expiry_dates_2022.get(next_month, "2023-01-18"))
        contract = f"{next_year}{next_month:02d}"
    else:
        expiry_date = this_month_expiry
        contract = f"2022{month:02d}"

    # 計算 Maturity (到期日 - 交易日，採日曆日) [cite: 28]
    maturity = (expiry_date - curr_date).days
    return contract, expiry_date, maturity

# 執行邏輯應用
df[['Contract', 'ContractExpiry Date', 'Maturity']] = df.apply(
    process_options_logic, axis=1, result_type='expand'
)

# --- D. 補齊其餘規範欄位 ---
# 建立每日檔名 File [cite: 26]
df['File'] = df['Date'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')
# 補入 Rf (2022 年央行升息後，建議設為約 1.2%) [cite: 29]
df['Rf'] = 0.012
# 轉換日期格式為 yyyy-mm-dd [cite: 25]
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
df['ContractExpiry Date'] = df['ContractExpiry Date'].dt.strftime('%Y-%m-%d')

# --- E. 輸出最終成果 [cite: 30] ---
final_cols = ['Date', 'File', 'S0', 'Contract', 'ContractExpiry Date', 'Maturity', 'Rf']
df_final = df[final_cols]

# 儲存檔案
df_final.to_csv("Index_學號_2022.csv", index=False)
print("2022 年度索引檔已產出成功！")
print(df_final.head(10)) # 展示前10筆資料 [cite: 48]

正在從 Yahoo Finance 抓取 2022 年加權指數資料...


/tmp/ipykernel_13163/407424974.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_s0 = yf.download("^TWII", start="2022-01-01", end="2022-12-31")
[*********************100%***********************]  1 of 1 completed


2022 年度索引檔已產出成功！
         Date                         File            S0 Contract  \
0  2022-01-03  OptionsDaily_2022_01_03.csv  18270.509766   202201   
1  2022-01-04  OptionsDaily_2022_01_04.csv  18526.349609   202201   
2  2022-01-05  OptionsDaily_2022_01_05.csv  18499.960938   202201   
3  2022-01-06  OptionsDaily_2022_01_06.csv  18367.919922   202201   
4  2022-01-07  OptionsDaily_2022_01_07.csv  18169.759766   202201   
5  2022-01-10  OptionsDaily_2022_01_10.csv  18239.380859   202201   
6  2022-01-11  OptionsDaily_2022_01_11.csv  18288.210938   202201   
7  2022-01-12  OptionsDaily_2022_01_12.csv  18375.400391   202201   
8  2022-01-13  OptionsDaily_2022_01_13.csv  18436.929688   202201   
9  2022-01-14  OptionsDaily_2022_01_14.csv  18403.330078   202201   

  ContractExpiry Date  Maturity     Rf  
0          2022-01-19        16  0.012  
1          2022-01-19        15  0.012  
2          2022-01-19        14  0.012  
3          2022-01-19        13  0.012  
4          2022-01

In [ ]:
import pandas as pd
import yfinance as yf

# 1. 抓取 2022 年 S0 (加權指數 ^TWII)
df_s0 = yf.download("^TWII", start="2022-01-01", end="2022-12-31")
df = df_s0[['Close']].reset_index()
df.columns = ['Date', 'S0']

# 2. 定義結算日 (2022年各月第三個週三)
expiry_dates_2022 = {
    1: "2022-01-19", 2: "2022-02-16", 3: "2022-03-16",
    4: "2022-04-20", 5: "2022-05-18", 6: "2022-06-15",
    7: "2022-07-20", 8: "2022-08-17", 9: "2022-09-21",
    10: "2022-10-19", 11: "2022-11-16", 12: "2022-12-21"
}

# 3. 精確利率邏輯：依據台銀 2022 年一般固定利率調整日
def get_exact_rf_2022(dt):
    if dt < pd.to_datetime("2022-03-21"): return 0.0079   # 1月基準
    elif dt < pd.to_datetime("2022-06-20"): return 0.0107  # 3月調升
    elif dt < pd.to_datetime("2022-09-26"): return 0.0122  # 6月調升
    elif dt < pd.to_datetime("2022-12-19"): return 0.0135  # 9月調升
    else: return 0.01475                                   # 12月調升

# 4. 資料建構邏輯
def process_data(row):
    curr_date = row['Date']
    this_expiry = pd.to_datetime(expiry_dates_2022[curr_date.month])

    if curr_date > this_expiry:
        next_m = curr_date.month + 1 if curr_date.month < 12 else 1
        year_label = 2022 if curr_date.month < 12 else 2023
        exp = pd.to_datetime(expiry_dates_2022.get(next_m, "2023-01-18"))
        con = f"{year_label}{next_m:02d}"
    else:
        exp = this_expiry
        con = f"2022{curr_date.month:02d}"

    return con, exp, (exp - curr_date).days

# 執行並補齊欄位
df[['Contract', 'ContractExpiry Date', 'Maturity']] = df.apply(process_data, axis=1, result_type='expand')
df['Rf'] = df['Date'].apply(get_exact_rf_2022)
df['File'] = df['Date'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')

# 輸出最終檔案
final_df = df[['Date', 'File', 'S0', 'Contract', 'ContractExpiry Date', 'Maturity', 'Rf']]
final_df.to_csv("Index_學號_2022.csv", index=False)
print("2022 索引檔已依台銀精確利率建構完成！")

/tmp/ipykernel_13163/353007033.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_s0 = yf.download("^TWII", start="2022-01-01", end="2022-12-31")
[*********************100%***********************]  1 of 1 completed


2022 索引檔已依台銀精確利率建構完成！


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import yfinance as yf
import datetime

# --- 1. 自動抓取 2022 年標的價格 (S0) ---
# 使用 auto_adjust=False 以取得原始收盤價，避免 yfinance 預設調整導致數據偏誤
print("正在抓取 2022 年台股加權指數 (^TWII) 原始收盤價...")
df_s0 = yf.download("^TWII", start="2022-01-01", end="2022-12-31", auto_adjust=False)
df = df_s0[['Close']].reset_index()
df.columns = ['Date', 'S0']

# --- 2. 設定 2022 年台指選擇權結算日 (每月第三個週三) ---
expiry_dates_2022 = {
    1: "2022-01-19", 2: "2022-02-16", 3: "2022-03-16",
    4: "2022-04-20", 5: "2022-05-18", 6: "2022-06-15",
    7: "2022-07-20", 8: "2022-08-17", 9: "2022-09-21",
    10: "2022-10-19", 11: "2022-11-16", 12: "2022-12-21"
}

# --- 3. 精確利率邏輯：台灣銀行一年期定期儲蓄存款－一般固定利率 ---
# 根據 2022 年央行升息與台銀公告調整日分段
def get_bot_fixed_rate_2022(dt):
    # 1月基準利率為 0.79%
    if dt < pd.to_datetime("2022-03-21"):
        return 0.0079
    # 3/21 升息後調升至 1.07%
    elif dt < pd.to_datetime("2022-06-20"):
        return 0.0107
    # 6/20 升息後調升至 1.22%
    elif dt < pd.to_datetime("2022-09-26"):
        return 0.0122
    # 9/26 升息後調升至 1.35%
    elif dt < pd.to_datetime("2022-12-19"):
        return 0.0135
    # 12/19 升息後調升至 1.475%
    else:
        return 0.01475

# --- 4. 核心邏輯：判定 Contract, Expiry Date, 並計算 Maturity ---
def process_options_logic(row):
    curr_date = row['Date']
    month = curr_date.month

    # 取得當月結算日
    this_month_expiry = pd.to_datetime(expiry_dates_2022[month])

    # 若今日已過結算日，則判定為「次月契約」
    if curr_date > this_month_expiry:
        next_month = month + 1 if month < 12 else 1
        next_year = 2022 if month < 12 else 2023
        # 取得下個月結算日 (若跨年則預設 2023-01-18)
        expiry_date = pd.to_datetime(expiry_dates_2022.get(next_month, "2023-01-18"))
        contract = f"{next_year}{next_month:02d}"
    else:
        expiry_date = this_month_expiry
        contract = f"2022{month:02d}"

    # 計算 Maturity (到期日 - 交易日，採日曆日) [cite: 28]
    maturity = (expiry_date - curr_date).days
    return contract, expiry_date, maturity

# --- 5. 執行運算與欄位整理 ---
df[['Contract', 'ContractExpiry Date', 'Maturity']] = df.apply(
    process_options_logic, axis=1, result_type='expand'
)
df['Rf'] = df['Date'].apply(get_bot_fixed_rate_2022) # 補入 Rf 並說明來源 [cite: 29]
df['File'] = df['Date'].dt.strftime('OptionsDaily_%Y_%m_%d.csv') # 建立每日檔名 [cite: 26]

# 統一日期格式為 yyyy-mm-dd [cite: 25, 69]
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
df['ContractExpiry Date'] = pd.to_datetime(df['ContractExpiry Date']).dt.strftime('%Y-%m-%d')

# --- 6. 輸出最終索引檔 ---
# 依照作業規範排序欄位：Date, File, S0, Contract, ContractExpiry Date, Maturity, Rf [cite: 8, 71]
final_cols = ['Date', 'File', 'S0', 'Contract', 'ContractExpiry Date', 'Maturity', 'Rf']
df_final = df[final_cols]

# 存檔為指定檔名規範 [cite: 38]
output_filename = "Index_學號_2022.csv"
df_final.to_csv(output_filename, index=False)

print(f"--- 2022 年度索引檔建構完成 ---")
print(f"檔案已儲存為: {output_filename}")
print(df_final.head(10)) # 展示前 10 筆資料供成果展示

[*********************100%***********************]  1 of 1 completed

正在抓取 2022 年台股加權指數 (^TWII) 原始收盤價...


--- 2022 年度索引檔建構完成 ---
檔案已儲存為: Index_學號_2022.csv
         Date                         File            S0 Contract  \
0  2022-01-03  OptionsDaily_2022_01_03.csv  18270.509766   202201   
1  2022-01-04  OptionsDaily_2022_01_04.csv  18526.349609   202201   
2  2022-01-05  OptionsDaily_2022_01_05.csv  18499.960938   202201   
3  2022-01-06  OptionsDaily_2022_01_06.csv  18367.919922   202201   
4  2022-01-07  OptionsDaily_2022_01_07.csv  18169.759766   202201   
5  2022-01-10  OptionsDaily_2022_01_10.csv  18239.380859   202201   
6  2022-01-11  OptionsDaily_2022_01_11.csv  18288.210938   202201   
7  2022-01-12  OptionsDaily_2022_01_12.csv  18375.400391   202201   
8  2022-01-13  OptionsDaily_2022_01_13.csv  18436.929688   202201   
9  2022-01-14  OptionsDaily_2022_01_14.csv  18403.330078   202201   

  ContractExpiry Date  Maturity      Rf  
0          2022-01-19        16  0.0079  
1          2022-01-19        15  0.0079  
2          2022-01-19        14  0.0079  
3          2022-01-19  

In [ ]:
import pandas as pd
import yfinance as yf
import datetime
from google.colab import drive

# --- 1. 掛載 Google Drive ---
# 執行後請點擊連結並授權，讓程式有權限將檔案存入妳的雲端
drive.mount('/content/drive')

# --- 2. 自動抓取 2022 年標的價格 (S0) ---
# 使用 auto_adjust=False 確保抓取「原始收盤價」，解決 FutureWarning 警告
print("正在從 Yahoo Finance 抓取 2022 年加權指數原始收盤價...")
df_s0 = yf.download("^TWII", start="2022-01-01", end="2022-12-31", auto_adjust=False)
df = df_s0[['Close']].reset_index()
df.columns = ['Date', 'S0']

# --- 3. 設定 2022 年台指選擇權結算日 (每月第三個週三) ---
expiry_dates_2022 = {
    1: "2022-01-19", 2: "2022-02-16", 3: "2022-03-16",
    4: "2022-04-20", 5: "2022-05-18", 6: "2022-06-15",
    7: "2022-07-20", 8: "2022-08-17", 9: "2022-09-21",
    10: "2022-10-19", 11: "2022-11-16", 12: "2022-12-21"
}

# --- 4. 精確利率邏輯：台灣銀行一年期定期儲蓄存款－一般固定利率 ---
# 完全依照妳提供的：1月 0.79%, 3月 1.07% 及其後升息路徑
def get_bot_fixed_rate_2022(dt):
    if dt < pd.to_datetime("2022-03-21"):
        return 0.0079   # 1月基準
    elif dt < pd.to_datetime("2022-06-20"):
        return 0.0107   # 3月調升
    elif dt < pd.to_datetime("2022-09-26"):
        return 0.0122   # 6月調升
    elif dt < pd.to_datetime("2022-12-19"):
        return 0.0135   # 9月調升
    else:
        return 0.01475  # 12月調升

# --- 5. 核心邏輯處理 ---
def process_options_logic(row):
    curr_date = row['Date']
    month = curr_date.month
    this_month_expiry = pd.to_datetime(expiry_dates_2022[month])

    if curr_date > this_month_expiry:
        next_month = month + 1 if month < 12 else 1
        next_year = 2022 if month < 12 else 2023
        expiry_date = pd.to_datetime(expiry_dates_2022.get(next_month, "2023-01-18"))
        contract = f"{next_year}{next_month:02d}"
    else:
        expiry_date = this_month_expiry
        contract = f"2022{month:02d}"

    maturity = (expiry_date - curr_date).days
    return contract, expiry_date, maturity

# 執行運算
df[['Contract', 'ContractExpiry Date', 'Maturity']] = df.apply(
    process_options_logic, axis=1, result_type='expand'
)
df['Rf'] = df['Date'].apply(get_bot_fixed_rate_2022)
df['File'] = df['Date'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
df['ContractExpiry Date'] = pd.to_datetime(df['ContractExpiry Date']).dt.strftime('%Y-%m-%d')

# --- 6. 整理格式並存入 Google Drive ---
final_cols = ['Date', 'File', 'S0', 'Contract', 'ContractExpiry Date', 'Maturity', 'Rf']
df_final = df[final_cols]

# 請將下面的 '學號' 替換成妳真實的學號
save_path = '/content/drive/My Drive/Index_學號_2022.csv'
df_final.to_csv(save_path, index=False)

print(f"--- 2022 年度索引檔建構完成 ---")
print(f"檔案已上傳至雲端硬碟路徑: {save_path}")
print(df_final.head(10))

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import drive
drive.flush_and_unmount() # 先解除掛載
drive.mount('/content/drive', force_remount=True) # 強制重新掛載

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
import os

# 1. 定義雲端硬碟資料夾路徑
# 注意：'My Drive' 中間有一個空格，這是 Google Drive 的預設名稱
folder_path = '/content/drive/MyDrive/'

# 檢查雲端硬碟是否真的存在
if os.path.exists(folder_path):
    # 執行存檔 (請記得把 '學號' 改成妳的真實學號)
    save_path = os.path.join(folder_path, 'Index_學號_2022.csv')
    df_final.to_csv(save_path, index=False)
    print(f"✅ 成功！檔案已存至雲端硬碟：{save_path}")
else:
    print("❌ 雲端硬碟路徑不存在，改為直接下載到電腦...")
    from google.colab import files
    df_final.to_csv('Index_學號_2022.csv', index=False)
    files.download('Index_學號_2022.csv')

✅ 成功！檔案已存至雲端硬碟：/content/drive/MyDrive/Index_學號_2022.csv
